In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
import os
import fiona

In [3]:
firepoint_1 = gpd.read_file("../data/hexagones_firepoint_1.geojson")
firepoint_2 = gpd.read_file("../data/hexagones_firepoint_2.geojson")
firepoint_3 = gpd.read_file("../data/hexagones_firepoint_3.geojson")
fire = gpd.read_file("../total_fire.geojson")

In [10]:
firepoint_1.head()

,hex_id,scale0,NDVI,NDMI,NDBI,NDSI,NDWI,osmnx,PasDeRoute,motorway,...,tree,grass,crops,shrub,flooded,built,bare,snow,sinister,geometry
0,871f91d93ffffff,0,0.489482,0.027680,-0.027680,-0.526345,-0.531443,None,98.156682,0.0,...,19.226831,0.908858,0.012801,52.662570,0.064004,2.355351,0.0,0.0,4.0,"POLYGON ((4.83447 45.99325, 4.83653 45.98116, ..."
1,871f91c58ffffff,1,0.489056,0.012780,-0.012780,-0.509756,-0.505684,None,95.492169,0.0,...,22.551891,3.323571,0.382020,13.243346,0.241946,16.210365,0.0,0.0,13.0,"POLYGON ((5.32436 46.05974, 5.32636 46.04762, ..."
2,871f91d35ffffff,2,0.539409,0.034183,-0.034183,-0.558490,-0.569783,None,97.275518,0.0,...,5.257099,6.382707,5.001279,42.862625,0.639550,6.152469,0.0,0.0,1.0,"POLYGON ((4.88902 46.22335, 4.89109 46.21128, ..."
3,871f91719ffffff,3,0.642989,0.106361,-0.106361,-0.551762,-0.613317,None,97.029326,0.0,...,11.704964,5.281198,0.330075,1.701155,0.380856,1.117177,0.0,0.0,3.0,"POLYGON ((5.73922 45.72338, 5.74115 45.71119, ..."
4,871f95600ffffff,4,0.482576,-0.035056,0.035056,-0.556619,-0.523738,None,96.868609,0.0,...,7.796524,6.045501,2.121677,16.308793,2.377301,25.217280,0.0,0.0,4.0,"POLYGON ((4.91318 46.44873, 4.91525 46.43670, ..."


In [20]:
firepoint_2.head()

,hex_id,scale0,NDVI,NDMI,NDBI,NDSI,NDWI,osmnx,PasDeRoute,motorway,...,tree,grass,crops,shrub,flooded,built,bare,snow,sinister,geometry
0,871f82a71ffffff,1292,0.772386,0.218191,-0.218191,-0.579142,-0.704039,None,97.222222,0.0,...,46.271271,15.477978,0.162663,0.838338,0.000000,1.564064,0.0,0.0,1.0,"POLYGON ((6.83497 47.35894, 6.83678 47.34692, ..."
1,871f82195ffffff,1293,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((5.99987 46.96160, 6.00180 46.94957, ..."
2,871f82726ffffff,1294,0.691865,0.242930,-0.242930,-0.460149,-0.598838,None,94.634085,0.0,...,55.472981,7.670991,0.100768,0.037788,0.000000,0.491246,0.0,0.0,4.0,"POLYGON ((6.24716 46.80847, 6.24905 46.79640, ..."
3,871f82813ffffff,1295,0.793321,0.246901,-0.246901,-0.606130,-0.739940,None,88.045397,0.0,...,85.977301,1.941992,0.113493,0.390921,0.012610,3.026482,0.0,0.0,0.0,"POLYGON ((6.05456 47.31775, 6.05649 47.30577, ..."
4,871f82720ffffff,1296,0.552192,0.230073,-0.230073,-0.292688,-0.425921,None,95.959215,0.0,...,29.481370,4.997482,0.226586,0.239174,0.037764,7.364048,0.0,0.0,6.0,"POLYGON ((6.27841 46.81237, 6.28029 46.80030, ..."


In [22]:
firepoint_3.head()

,hex_id,scale0,NDVI,NDMI,NDBI,NDSI,NDWI,osmnx,PasDeRoute,motorway,...,tree,grass,crops,shrub,flooded,built,bare,snow,sinister,geometry
0,871fb409bffffff,3019,0.494511,0.096142,-0.096142,-0.400418,-0.462165,None,95.551307,0.0,...,13.601020,0.662843,0.599108,5.659656,0.484385,59.273423,1.26195,0.0,9.0,"POLYGON ((1.96396 49.00566, 1.94996 48.99882, ..."
1,871fb6b21ffffff,3020,0.824315,0.348412,-0.348412,-0.571698,-0.766726,None,96.634799,0.0,...,95.321861,0.586361,0.000000,0.012747,0.000000,0.229446,0.00000,0.0,2.0,"POLYGON ((1.68115 48.73047, 1.66721 48.72355, ..."
2,871fb44eaffffff,3021,0.370989,-0.014092,0.014092,-0.452930,-0.432540,None,95.788668,0.0,...,2.922409,0.625319,2.935171,68.746810,0.306279,11.625830,0.00000,0.0,5.0,"POLYGON ((1.79931 48.82976, 1.78535 48.82288, ..."
3,871fb4451ffffff,3022,0.380476,-0.023379,0.023379,-0.481154,-0.449512,None,98.610403,0.0,...,0.000000,0.382458,0.293218,72.501275,0.025497,7.878633,0.00000,0.0,0.0,"POLYGON ((1.81496 48.90547, 1.80098 48.89860, ..."
4,871fb6a1dffffff,3023,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((1.84658 48.51277, 1.83269 48.50585, ..."


In [12]:
fire.head

<bound method NDFrame.head of                hex_id raison_sortie           date_debut             date_fin  \
0     871f91c4dffffff      NATURELS  2018-01-05 17:41:19  2018-01-05 18:45:21   
1     871f91c6effffff      NATURELS  2018-01-06 17:56:49  2018-01-06 19:37:07   
2     871f910d6ffffff      NATURELS  2018-01-12 08:43:15  2018-01-12 09:36:25   
3     871f91c9bffffff      NATURELS  2018-01-15 17:55:58  2018-01-15 18:59:14   
4     871f910e3ffffff      NATURELS  2018-01-19 22:57:16  2018-01-20 00:21:51   
...               ...           ...                  ...                  ...   
8564  871fb4684ffffff      NATURELS     09/09/2023 12:35     09/09/2023 13:32   
8565  871fb4793ffffff      NATURELS     09/09/2023 13:31     09/09/2023 13:51   
8566  871fb4795ffffff      NATURELS     10/09/2023 09:34     10/09/2023 11:04   
8567  871fb478bffffff      NATURELS     10/09/2023 14:54     10/09/2023 17:03   
8568  871fb4413ffffff      NATURELS     10/09/2023 17:04     10/09/2023 18:48  

In [4]:
hex_id1 = set(firepoint_3['hex_id'])
hex_id2 = set(fire['hex_id'])

common_hex_ids = hex_id1.intersection(hex_id2)

unique_to_dataset1 = hex_id1.difference(hex_id2)
unique_to_dataset2 = hex_id2.difference(hex_id1)

common_departments = fire[fire['hex_id'].isin(common_hex_ids)]['departement'].unique()
unique_departments_dataset2 = fire[fire['hex_id'].isin(unique_to_dataset2)]['departement'].unique()

print(f"Nombre total de hex_id dans Dataset 1 : {len(hex_id1)}")
print(f"Nombre total de hex_id dans Dataset 2 : {len(hex_id2)}")
print(f"Nombre d'hex_id communs : {len(common_hex_ids)}")
print(f"Départements associés aux hex_id communs : {list(common_departments)}")
print(f"Hex_id uniques au Dataset 1 : {unique_to_dataset1}")
print(f"Hex_id uniques au Dataset 2 : {unique_to_dataset2}")
print(f"Départements associés aux hex_id uniques au Dataset 2 : {list(unique_departments_dataset2)}")

Nombre total de hex_id dans Dataset 1 : 576
Nombre total de hex_id dans Dataset 2 : 1913
Nombre d'hex_id communs : 417
Départements associés aux hex_id communs : ['departement-78-yvelines']
Hex_id uniques au Dataset 1 : {'871fb452effffff', '871fb4685ffffff', '871fb44adffffff', '871fb6a25ffffff', '871fb6ae3ffffff', '871fb4433ffffff', '871fb6a34ffffff', '871fb6b5effffff', '871fb40b0ffffff', '871fb4450ffffff', '871fb450bffffff', '871fb444bffffff', '871fb44e5ffffff', '871fb6b2effffff', '871fb6b04ffffff', '871fb479bffffff', '871fb44d9ffffff', '871fb6a1bffffff', '871fb6ae5ffffff', '871fb4694ffffff', '871fb4451ffffff', '871fb4472ffffff', '871fb4411ffffff', '871fb4096ffffff', '871fb6a46ffffff', '871fb4724ffffff', '871fb4082ffffff', '871fb44c3ffffff', '871fb4483ffffff', '871fb4716ffffff', '871fb6a31ffffff', '871fb441effffff', '871fb458dffffff', '871fb6a60ffffff', '871fb6b2affffff', '871fb444cffffff', '871fb6a6effffff', '871fb4446ffffff', '871fb6a55ffffff', '871fb6ae2ffffff', '871fb6b59ffffff', 

In [35]:
firepoint_2 = firepoint_2.drop(columns=['geometry'], errors='ignore')

meteo_doubs = meteo_doubs.merge(firepoint_2, on='hex_id', how='left')

print(meteo_doubs.head())

            hex_id  id_meteo  duree_minutes       date           departement  \
0  871f82a2effffff       155           69.0 2016-01-03  departement-25-doubs   
1  871f82c08ffffff      2135           94.0 2016-02-06  departement-25-doubs   
2  871f82041ffffff      2107           53.0 2016-02-06  departement-25-doubs   
3  871f80793ffffff      2228           63.0 2015-06-06  departement-25-doubs   
4  871f82c73ffffff      2274           49.0 2016-02-09  departement-25-doubs   

   centroid_lon  centroid_lat  hour_sin      hour_cos     month_sin  ...  \
0      6.706744     47.355752 -0.866025 -5.000000e-01  5.000000e-01  ...   
1      5.793049     47.057733 -1.000000 -1.836970e-16  8.660254e-01  ...   
2      6.530495     47.040158 -1.000000 -1.836970e-16  8.660254e-01  ...   
3      6.824442     47.534949  0.258819  9.659258e-01  1.224647e-16  ...   
4      5.868218     47.085815 -0.707107 -7.071068e-01  8.660254e-01  ...   

      water       tree      grass     crops      shrub   flood

In [36]:
meteo_doubs.to_file("../datasets_par_departement/meteo_doubs.geojson", driver="GeoJSON")

In [38]:
meteo_doubs.head()

,hex_id,id_meteo,duree_minutes,date,departement,centroid_lon,centroid_lat,hour_sin,hour_cos,month_sin,...,tree,grass,crops,shrub,flooded,built,bare,snow_y,sinister,geometry
0,871f82a2effffff,155,69.0,2016-01-03,departement-25-doubs,6.706744,47.355752,-0.866025,-5.000000e-01,5.000000e-01,...,34.156121,24.345320,0.413482,1.052500,0.025060,2.744017,0.000000,0.0,0.0,"POLYGON ((6.69003 47.35985, 6.69187 47.34784, ..."
1,871f82c08ffffff,2135,94.0,2016-02-06,departement-25-doubs,5.793049,47.057733,-1.000000,-1.836970e-16,8.660254e-01,...,18.742886,8.169976,1.214114,16.087012,0.189705,8.739092,0.000000,0.0,2.0,"POLYGON ((5.77646 47.06172, 5.77842 47.04971, ..."
2,871f82041ffffff,2107,53.0,2016-02-06,departement-25-doubs,6.530495,47.040158,-1.000000,-1.836970e-16,8.660254e-01,...,29.604520,13.421218,0.075330,0.740741,0.037665,2.109228,0.000000,0.0,2.0,"POLYGON ((6.51386 47.04426, 6.51572 47.03221, ..."
3,871f80793ffffff,2228,63.0,2015-06-06,departement-25-doubs,6.824442,47.534949,0.258819,9.659258e-01,1.224647e-16,...,6.781782,0.250250,0.087588,0.588088,0.050050,72.972973,0.775776,0.0,27.0,"POLYGON ((6.80769 47.53905, 6.80951 47.52705, ..."
4,871f82c73ffffff,2274,49.0,2016-02-09,departement-25-doubs,5.868218,47.085815,-0.707107,-7.071068e-01,8.660254e-01,...,33.143724,7.976236,1.327266,4.689673,0.265453,5.018329,0.000000,0.0,1.0,"POLYGON ((5.85161 47.08981, 5.85357 47.07780, ..."
